In [1]:
import os
import pandas as pd
import numpy as np

import json
from tqdm import tqdm
import random
import matplotlib.pyplot as plt
import shutil
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
import base64
import seaborn as sns

import tensorflow as tf

warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)

#### Loading finbert model

In [2]:
# Step 15: Generate FinBERT embeddings for daily aggregated news
print("Installing and loading FinBERT model...")

# Install transformers if not already installed
import subprocess
import sys

try:
    from transformers import AutoTokenizer, AutoModel
except ImportError:
    print("Installing transformers and torch...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "torch"])
    from transformers import AutoTokenizer, AutoModel

import torch

# Load FinBERT model (financial domain BERT)
model_name = "ProsusAI/finbert"
print(f"Loading model: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)
finbert_model = AutoModel.from_pretrained(model_name)

# Move model to CPU/GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
finbert_model.to(device)
finbert_model.eval()

print(f"Model loaded on device: {device}")

Installing and loading FinBERT model...
Loading model: ProsusAI/finbert


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 31896.46it/s]
[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on device: cpu


#### Data pipeline

In [12]:
Market = 'SP500'     # Can take values ['SP500', 'FTSE', 'STOXX', 'DAX', 'CAC', 'CBU0'] 

In [13]:
# Step 13: Load news data and create FinBERT embeddings
print("Loading full news dataset...")
news_df = pd.read_csv('Data/News/Guardian Raw/Market_specific_news.csv')
news_df = news_df.loc[news_df['market'].apply(lambda x: Market in x)].reset_index(drop=True)
print(f"Total news articles: {len(news_df)}")
news_df.head()

Loading full news dataset...
Total news articles: 31216


,id,type,sectionId,sectionName,webPublicationDate,webTitle,webUrl,apiUrl,tags,references,...,pillarId,pillarName,headline,trailText,bodyText_200,wordcount,byline,market,search_query,_fetch_page
0,media/2006/jan/03/1,article,media,Media,2006-01-03T08:58:10Z,MediaGuardian Briefing: January 3,https://www.theguardian.com/media/2006/jan/03/1,https://content.guardianapis.com/media/2006/ja...,"[{""id"": ""media/media"", ""type"": ""keyword"", ""sec...",[],...,pillar/news,News,Today's media stories from the papers,<p>ON MEDIAGUARDIAN.CO.UK TODAY:<br />Tony Bla...,THE GUARDIAN Prime minister Tony Blair has rel...,470,NaN,SP500,"""S&P 500"" OR ""S&P500"" OR ""S&P 500 index"" OR ""S...",1
1,business/2006/jan/03/dowjones.citynews,article,business,Business,2006-01-03T17:30:30Z,Dow Jones appoints new chief executive,https://www.theguardian.com/business/2006/jan/...,https://content.guardianapis.com/business/2006...,"[{""id"": ""business/dowjones"", ""type"": ""keyword""...",[],...,pillar/news,News,Dow Jones appoints new chief executive,"<p><strong>5.30pm: </strong>Dow Jones, the pub...","Dow Jones, the publisher of the Wall Street Jo...",219,"Chris Tryhorn, City correspondent",SP500,"""S&P 500"" OR ""S&P500"" OR ""S&P 500 index"" OR ""S...",1
2,media/2006/jan/04/citynews.business,article,media,Media,2006-01-04T07:23:25Z,Viacom joins the urge to demerge,https://www.theguardian.com/media/2006/jan/04/...,https://content.guardianapis.com/media/2006/ja...,"[{""id"": ""media/media"", ""type"": ""keyword"", ""sec...","[{""id"": ""reuters-stock-ric/VIAB.O"", ""type"": ""r...",...,pillar/news,News,Viacom joins the urge to demerge,<p>The days of the sprawling media conglomerat...,The days of the sprawling media conglomerate c...,231,David Teather in New York,SP500,"""S&P 500"" OR ""S&P500"" OR ""S&P 500 index"" OR ""S...",1
3,media/2006/jan/04/pressandpublishing,article,media,Media,2006-01-04T09:03:41Z,MediaGuardian Briefing: January 4,https://www.theguardian.com/media/2006/jan/04/...,https://content.guardianapis.com/media/2006/ja...,"[{""id"": ""media/pressandpublishing"", ""type"": ""k...",[],...,pillar/news,News,Today's media stories from the papers,<p>ON MEDIAGUARDIAN.CO.UK TODAY:<br />Channel ...,THE GUARDIAN Toothpaste makers told to drop ad...,533,NaN,SP500,"""S&P 500"" OR ""S&P500"" OR ""S&P 500 index"" OR ""S...",1
4,business/2006/jan/05/ftse.stockmarkets1,article,business,Business,2006-01-05T01:39:43Z,Market forces: HBOS misses out in run on the b...,https://www.theguardian.com/business/2006/jan/...,https://content.guardianapis.com/business/2006...,"[{""id"": ""business/ftse"", ""type"": ""keyword"", ""s...","[{""id"": ""reuters-index-ric/.FTSE"", ""type"": ""re...",...,pillar/news,News,HBOS misses out in run on the banks,<p><strong> Market forces:</strong>Britain's f...,"Britain's fourth biggest bank by market value,...",997,NaN,SP500 | FTSE | CBU0,"""S&P 500"" OR ""S&P500"" OR ""S&P 500 index"" OR ""S...",1


In [14]:
# Convert webPublicationDate to datetime
news_df['webPublicationDate'] = pd.to_datetime(news_df['webPublicationDate'], errors='coerce')
news_df['date'] = news_df['webPublicationDate'].dt.date

# Display sample
print("\nNews data sample:")
print(news_df[['date', 'webTitle', 'headline']].head())
print(f"\nNews date range: {news_df['date'].min()} to {news_df['date'].max()}")



# Step 14: Aggregate news headlines by day
print("Aggregating news headlines by day...")

# Fill NaN headlines with empty string
news_df['headline'] = news_df['headline'].fillna('')
news_df['webTitle'] = news_df['webTitle'].fillna('')

# Aggregate headlines for each day
daily_news = news_df.groupby('date').agg({
    'headline': lambda x: ' '.join(x.dropna().astype(str)),
    'webTitle': lambda x: ' '.join(x.dropna().astype(str)),
    'id': 'count'  # Count of articles per day
}).reset_index()

daily_news.columns = ['date', 'daily_headlines', 'daily_titles', 'article_count']
daily_news['date'] = pd.to_datetime(daily_news['date'])

# Combine headlines and titles for better context
daily_news['aggregated_text'] = daily_news['daily_headlines'] + ' ' + daily_news['daily_titles']

print(f"Total unique days with news: {len(daily_news)}")
print("\nDaily news aggregation sample:")
print(daily_news[['date', 'article_count']].head(10))



# Function to generate embeddings
def get_finbert_embedding(text, max_length=512):
    """Generate FinBERT embedding for text"""
    if not text or len(text.strip()) == 0:
        return np.zeros(768)  # Return zero vector for empty text
    
    # Truncate text if too long
    text = text[:2000]  # Limit to 2000 chars before tokenization
    
    # Tokenize
    inputs = tokenizer(
        text,
        add_special_tokens=True,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    # Generate embedding
    with torch.no_grad():
        outputs = finbert_model(**inputs.to(device))
        # Use [CLS] token embedding (first token)
        embedding = outputs.last_hidden_state[0, 0, :].cpu().numpy()
    
    return embedding

# Generate embeddings for each day
print("\nGenerating FinBERT embeddings for daily news (this may take a few minutes)...")
daily_embeddings = []

for idx, row in daily_news.iterrows():
    if idx % 100 == 0:
        print(f"  Processing day {idx + 1}/{len(daily_news)}...")
    
    # Here we can specify what to use for embedding: aggregated_text, daily_headlines, or daily_titles
    embedding = get_finbert_embedding(row['daily_titles'])  # Can change this to 'aggregated_text' or 'daily_headlines' as needed
    daily_embeddings.append(embedding)

# Convert to dataframe
embeddings_array = np.array(daily_embeddings)
embedding_cols = [f'news_embedding_{i}' for i in range(embeddings_array.shape[1])]
news_embeddings_df = pd.DataFrame(embeddings_array, columns=embedding_cols)
news_embeddings_df['date'] = daily_news['date'].values
news_embeddings_df.to_csv(f'Data/{Market}_daily_news_embeddings.csv', index=False)

print(f"\nEmbeddings shape: {embeddings_array.shape}")
print(f"Sample embedding stats - mean: {embeddings_array.mean():.4f}, std: {embeddings_array.std():.4f}")



News data sample:
         date                                           webTitle  \
0  2006-01-03                  MediaGuardian Briefing: January 3   
1  2006-01-03             Dow Jones appoints new chief executive   
2  2006-01-04                   Viacom joins the urge to demerge   
3  2006-01-04                  MediaGuardian Briefing: January 4   
4  2006-01-05  Market forces: HBOS misses out in run on the b...   

                                 headline  
0   Today's media stories from the papers  
1  Dow Jones appoints new chief executive  
2        Viacom joins the urge to demerge  
3   Today's media stories from the papers  
4     HBOS misses out in run on the banks  

News date range: 2006-01-03 to 2026-08-31
Aggregating news headlines by day...
Total unique days with news: 7091

Daily news aggregation sample:
        date  article_count
0 2006-01-03              2
1 2006-01-04              2
2 2006-01-05              4
3 2006-01-06              4
4 2006-01-07          